# Interactive Prototype

Goal: Create a functional MVP for field route planning and management.

Stages:
1. Region Input & Sub-Division
2. Automatic Target Search & Routing
3. Manual Adjustment
4. Select & Execute Plans

## Region Input & Sub-Division

- Get region outline
- Divide into work cells
- Tentative plan for depot locations
- 

### Dummy Region Shapefile

At this point, we do not actually have a shapefile of the target region. The following two Jupyter cells will generate one using the outline of the orthophoto we have created. 

Load this as the "shapefile" which will define our working region. 

In [1]:
region_image_path = '../input/IGNORE_Brewster-2024-all-orthophoto-UTM-32613.tif'
region_contour_shapefile = '../input/interactive_proto/region_contour.shp'
region_contour_geojson = '../input/interactive_proto/region_contour.geojson'


region_crs = 32613 # Use this everywhere for consistency
visualization_crs = 4326 # Use this when we need leaflet visualizations
simplification_tolerance = 5


In [2]:
import sys
import geopandas as gpd
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

### Create Voronoi Partitioning, Solve for Depots

- [x] Display region outline
- [x] Display region partition cells, centroids
- [x] Find and indicate depot locations
- [ ] Plan routes (pre-target adjustment CV) 

In [3]:
target_area_acres = 0.5
# target_area_acres = 1.5
# target_area_acres = 2.5

target_area_sqm = target_area_acres * 4046.86
max_iterations = 15 # Cycles to find improved partition

voronoi_partition_filename = '../input/interactive_proto/voronoi_partition.geojson'
voronoi_centroids_filename = '../input/interactive_proto/voronoi_centroids.geojson'

# Depot placement parameters
depot_radius = 225  # Max distance a depot can cover
depots_filename = '../input/interactive_proto/depot_points.geojson'

In [4]:
from plant_search.region_partition import centroidal_voronoi_tessellation
from macro_planning.depot_placement import find_depots, assign_cells_to_depot

region_outline_gdf = gpd.read_file(region_contour_shapefile)
simplified_polygon = region_outline_gdf.geometry.iloc[0]
# print(loaded_gdf.crs)
num_cells = int(simplified_polygon.area / target_area_sqm) # How many cells to generate

# Divide region into voronoi cells
cell_gdf = centroidal_voronoi_tessellation(simplified_polygon, num_cells, max_iterations)

# Find depots to cover all cells
grid_density = 4
depots_gdf = find_depots(depot_radius, cell_gdf, region_outline_gdf, grid_density)

# for depot_id, depot in depots_gdf.iterrows():
#     print(f'{depot_id}: {depot["geometry"]}')

cell_gdf = assign_cells_to_depot(depots_gdf, cell_gdf)


# for depot_id, depot in updated_cell_gdf.iterrows():
#     print(f'{depot_id}: {depot["closest_depot"]}')

Reached maximum iterations without full convergence.


In [5]:
from macro_planning.depot_placement import minimum_enclosing_circle

# target_depot = 'depot_99'

# dict_of_groups = {
#     key: group
#     for key, group in cell_gdf.groupby('closest_depot')
# }

# # print(dict_of_groups.keys())

# region_cells = dict_of_groups[target_depot]
# center, radius = minimum_enclosing_circle(region_cells)

# print(radius)
depots_gdf

,geometry,depot_radius,depot_id,min_enclosing_rad
0,POINT (634319.763 3347230.432),225,depot_25,208.720606
1,POINT (634694.32 3347197.849),225,depot_93,203.713996
2,POINT (634502.353 3347165.975),225,depot_145,218.517936


#### Write Data to Files

1. Region cells
2.  Region cell centroids
3. Depot locations

In [6]:
cell_gdf_4326 = cell_gdf.copy().to_crs(visualization_crs)
# print(cell_gdf_4326.crs)

# Create a copy with only the 'geometry' column (Voronoi polygons)
voronoi_gdf = cell_gdf_4326.copy().drop(columns=["cell_centroid"])
voronoi_gdf.to_crs(visualization_crs, inplace=True)
voronoi_gdf.to_file(voronoi_partition_filename, driver="GeoJSON")

# Create a copy with only the 'cell_centroid' column and set it as the active geometry
centroid_gdf = cell_gdf_4326.copy().drop(columns=["geometry"])
centroid_gdf.set_geometry("cell_centroid", inplace=True)

centroid_gdf.set_crs(region_crs, inplace=True)  # Reset the CRS explicitly
centroid_gdf.to_crs(visualization_crs, inplace=True)  # Reset the CRS explicitly
centroid_gdf.to_file(voronoi_centroids_filename, driver="GeoJSON")

# Write Depot locations to file
depots_gdf.to_crs(visualization_crs, inplace=True)
depots_gdf.to_file(depots_filename, driver="GeoJSON")


#### Data Interaction with Leaflet

In [7]:
from ipyleaflet import Circle, CircleMarker, LayerGroup

def create_depot_layers(depot_data):
    depot_layers = []

    for feature in depot_data["features"]:
        depot_plots = [] # Hold range, centerpoint circles
        coords = feature["geometry"]["coordinates"]
        properties = feature["properties"]
        depot_radius = properties.get("depot_radius", 0)  # Default to 0 if missing
        depot_id = properties.get("depot_id", "Unknown ID")
        depot_name = f'Depot {depot_id}'

        range_circle = Circle(
            location=[coords[1], coords[0]],  # GeoJSON uses (lon, lat), Folium expects (lat, lon)
            radius=depot_radius,  # Circle radius in meters
            color='black', fill=False, fill_color='#3366cc',
            fill_opacity=0.05, weight=1,
            tooltip=f"Depot ID: {depot_id}\nRadius: {depot_radius}m"
        )
        
        center_circle = CircleMarker(
            location=[coords[1], coords[0]],  # GeoJSON uses (lon, lat), Folium expects (lat, lon)
            radius=5,  # Circle radius in meters
            color='black', fill=True, fill_color='red',
            fill_opacity=0.9, weight=1,
            tooltip=f"Depot ID: {depot_id}\nRadius: {depot_radius}m"
        )

        depot_layergroup = LayerGroup(
            layers=(range_circle, center_circle),
            name=depot_name
        )
        depot_layers.append(depot_layergroup)
    
    return depot_layers

In [8]:
import geopandas as gpd
from ipyleaflet import (
    Map, GeoJSON, GeoData, Circle, LayerGroup,
    LayersControl, ScaleControl
)
from shapely.geometry import mapping, shape
import json

# Load the GeoJSON region outline
with open(region_contour_geojson, "r") as f:
    region_contour_data = json.load(f)
region_geometry = shape(region_contour_data['features'][0]['geometry'])
region_center = region_geometry.centroid

# Load Voronoi polygons
with open(voronoi_partition_filename, "r") as f:
    voronoi_data = json.load(f)

# Load centroids
with open(voronoi_centroids_filename, "r") as f:
    centroid_data = json.load(f)

# Load depot locations
with open(depots_filename, "r") as f:
    depot_data = json.load(f)

# print(region_contour_data)
print(voronoi_data)
# print(centroid_data)
# print(depot_data)




m = Map(center=(region_center.y, region_center.x), zoom=16)

# Add the region border to the map
region_layer = GeoJSON(
    data=region_contour_data, 
    style={'color': 'green', 'fillOpacity': 0.2, 'weight': 3},
    name=region_contour_data['name'])
m.add_layer(region_layer)

# Add Voronoi polygons
voronoi_layer = GeoJSON(
    data=voronoi_data, 
    style={'color': 'blue', 'fillColor': 'lightblue', 'opacity': 0.5, 'weight': 2},
    name=voronoi_data['name'])
m.add_layer(voronoi_layer)

# Add centroids
centroid_layer = GeoJSON(
    data=centroid_data, 
    style={'color': 'black', 'radius':3, 'fillColor': '#3366cc', 'opacity':0.5, 'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6},
    hover_style={'fillColor': 'red' , 'fillOpacity': 0.2},
    point_style={'radius': 3, 'color': 'red', 'fillOpacity': 0.8, 'fillColor': 'blue', 'weight': 3},
    name=centroid_data['name'])
centroid_layer.visible = False  # Set layer to hidden
m.add_layer(centroid_layer)

# Plot depot circles
depot_layers = create_depot_layers(depot_data)
for depot_layer in depot_layers:
    m.add(depot_layer)


m.add_control(LayersControl(position='topright'))
m.add(ScaleControl(position='bottomleft'))
m # Display the map

{'type': 'FeatureCollection', 'name': 'voronoi_partition', 'crs': {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}}, 'features': [{'type': 'Feature', 'properties': {'cell_id': 0, 'associated_depots': ['depot_25'], 'closest_depot': 'depot_25'}, 'geometry': {'type': 'Polygon', 'coordinates': [[[-103.60505321349447, 30.249579365713913], [-103.60511468519341, 30.24929747748494], [-103.60547253014781, 30.249251569715486], [-103.60550211222898, 30.24957322359244], [-103.60543470121637, 30.249725154769347], [-103.60505321349447, 30.249579365713913]]]}}, {'type': 'Feature', 'properties': {'cell_id': 1, 'associated_depots': ['depot_25'], 'closest_depot': 'depot_25'}, 'geometry': {'type': 'Polygon', 'coordinates': [[[-103.60505321349447, 30.249579365713913], [-103.60492691327485, 30.249637665329875], [-103.60464276087647, 30.249510650046037], [-103.60462365769715, 30.249257013385805], [-103.60466168144544, 30.249210434972877], [-103.6050149131783, 30.24920847100854], [-10

Map(center=[30.24893165719689, -103.6019209136208], controls=(ZoomControl(options=['position', 'zoom_in_text',…

## Show Routes from each Depot

- Show cells associated with each depot
- Show routes to cover all cells from each depot

In [9]:
cell_group_gdfs = [x for _, x in cell_gdf.groupby('closest_depot')]

print(len(cell_group_gdfs))
print(type(cell_group_gdfs[0]))

# cell_gdf_dict = cell_gdf.groupby('closest_depot').apply().to_dict()
dict_of_groups = {
    key: group
    for key, group in cell_gdf.groupby('closest_depot')
}

# print(dict_of_groups)

# key1 = list(dict_of_groups.keys())[0]
# print(len(dict_of_groups.keys()))
# print(dict_of_groups[key1])
# print(type(dict_of_groups[key1][0]))

3
<class 'geopandas.geodataframe.GeoDataFrame'>


In [10]:
from ipyleaflet import Choropleth, GeoJSON
import matplotlib as plt

colors = plt.cm.tab20(range(len(depots_gdf)))  # Use tab20 colormap for up to 20 depots
colors = ['red', 'yellow', 'orange', 'green', 'blue', 'purple']
depot_colors = {depot['depot_id']: colors[i] for i, depot in depots_gdf.iterrows()}
print(depot_colors)

cell_coloring = dict(zip(cell_gdf['cell_id'], cell_gdf['closest_depot']))

print(cell_coloring)

feature = voronoi_data['features'][0]
print(feature['properties']['closest_depot'])

def color_cells(feature):
    return {
        'fillColor': depot_colors[str(feature['properties']['closest_depot'])],
        'color': depot_colors[feature['properties']['closest_depot']],
        'opacity': 0.99,
        'weight': 2,
    }

m2 = Map(center=(region_center.y, region_center.x), zoom=16)

voronoi_layer = GeoJSON(
    data=voronoi_data, 
    # style={'color': 'red', 'fillColor': 'lightblue', 'opacity': 0.5, 'weight': 2},
    style_callback=color_cells,
    name=voronoi_data['name'])
m2.add_layer(voronoi_layer)

m2

{'depot_25': 'red', 'depot_93': 'yellow', 'depot_145': 'orange'}
{0: 'depot_25', 1: 'depot_25', 2: 'depot_25', 3: 'depot_25', 4: 'depot_25', 5: 'depot_25', 6: 'depot_25', 7: 'depot_25', 8: 'depot_25', 9: 'depot_25', 10: 'depot_25', 11: 'depot_25', 12: 'depot_25', 13: 'depot_25', 14: 'depot_25', 15: 'depot_25', 16: 'depot_25', 17: 'depot_25', 18: 'depot_25', 19: 'depot_25', 20: 'depot_25', 21: 'depot_25', 22: 'depot_25', 23: 'depot_25', 24: 'depot_25', 25: 'depot_25', 26: 'depot_25', 27: 'depot_25', 28: 'depot_25', 29: 'depot_25', 30: 'depot_25', 31: 'depot_145', 32: 'depot_25', 33: 'depot_25', 34: 'depot_145', 35: 'depot_25', 36: 'depot_25', 37: 'depot_145', 38: 'depot_25', 39: 'depot_25', 40: 'depot_25', 41: 'depot_25', 42: 'depot_145', 43: 'depot_25', 44: 'depot_25', 45: 'depot_25', 46: 'depot_145', 47: 'depot_145', 48: 'depot_145', 49: 'depot_145', 50: 'depot_25', 51: 'depot_145', 52: 'depot_145', 53: 'depot_145', 54: 'depot_145', 55: 'depot_145', 56: 'depot_145', 57: 'depot_145', 5

Map(center=[30.24893165719689, -103.6019209136208], controls=(ZoomControl(options=['position', 'zoom_in_text',…

In [22]:


def depot_selection_layers(depot_data, cell_data):
    depot_layers = {} # dict, where key is depot_id

    for feature in depot_data["features"]:
        depot_plots = [] # Hold range, centerpoint circles
        coords = feature["geometry"]["coordinates"]
        properties = feature["properties"]
        depot_radius = int(properties.get("depot_radius", 0))  # Default to 0 if missing
        min_encl_radius = int(properties.get("min_enclosing_rad", 0))  # Default to 0 if missing
        valid_depot_range = (depot_radius-min_encl_radius) if min_encl_radius>0 else 0

        depot_id = properties.get("depot_id", "Unknown ID")
        depot_name = f'Depot {depot_id}'

        # Find cells associated with each depot for coloration
        associated_cells = cell_data.copy()
        associated_cells['features'] = [feature for feature in cell_data['features']
                                        if feature['properties']['closest_depot'] == depot_id]

        # Add highlight to cells in depot range
        cells_layer = GeoJSON(
            data=associated_cells, 
            style={'color': 'red', 'fillColor': 'lightblue', 'opacity': 0.5, 'weight': 2},
            # style_callback=color_cells,
            name=associated_cells['name'])


        # Maximum range of depot
        range_circle = Circle(
            location=[coords[1], coords[0]],  # GeoJSON uses (lon, lat), Folium expects (lat, lon)
            radius=depot_radius,  # Circle radius in meters
            color='black', fill=False, fill_color='#3366cc',
            fill_opacity=0.05, weight=1,
            tooltip=f"Depot ID: {depot_id}\nRadius: {depot_radius}m"
        )

        # Plot our minimum enclosing circle
        min_enclosing_circle = Circle(
            location=[coords[1], coords[0]],  # GeoJSON uses (lon, lat), Folium expects (lat, lon)
            radius=min_encl_radius,  # Circle radius in meters
            color='green', fill=False, fill_color='#3366cc',
            fill_opacity=0.05, weight=1, opacity=0.15,
            tooltip=f"Depot ID: {depot_id}\nRadius: {min_encl_radius}m"
        )
        
        # All valid depot placements to cover all cells
        valid_depots_circle = Circle(
            location=[coords[1], coords[0]],  # GeoJSON uses (lon, lat), Folium expects (lat, lon)
            radius=valid_depot_range,  # Circle radius in meters
            color='green', fill=True, fill_color='green',
            fill_opacity=0.1, weight=1, opacity=0.35,
            tooltip=f"Valid depot range: {depot_id}\nRadius: {valid_depot_range}m"
        )

        center_circle = CircleMarker(
            location=[coords[1], coords[0]],  # GeoJSON uses (lon, lat), Folium expects (lat, lon)
            radius=5,  # Circle radius in meters
            color='black', fill=True, fill_color='red',
            fill_opacity=0.9, weight=1,
            tooltip=f"Depot ID: {depot_id}\nRadius: {depot_radius}m"
        )

        depot_layergroup = LayerGroup(
            layers=(cells_layer, range_circle, min_enclosing_circle, 
                    valid_depots_circle, center_circle),
            name=depot_name
        )
        depot_layers[depot_id] = depot_layergroup
    
    return depot_layers

In [23]:
from ipyleaflet import Choropleth, GeoJSON, WidgetControl
from ipywidgets import Select, Dropdown
import matplotlib as plt
from shapely.geometry import mapping, shape
import json


# Load Data for mapping
# =====================

# Load the GeoJSON region outline
with open(region_contour_geojson, "r") as f:
    region_contour_data = json.load(f)
region_geometry = shape(region_contour_data['features'][0]['geometry'])
region_center = region_geometry.centroid

# Load Voronoi cells
with open(voronoi_partition_filename, "r") as f:
    voronoi_data = json.load(f)

# Load depot locations
with open(depots_filename, "r") as f:
    depot_data = json.load(f)



# Set up interactive layer selections
# ===================================
all_depot_layers = depot_selection_layers(depot_data, voronoi_data) # Dict of layer instances
list_depots = list(all_depot_layers.keys())

# Depot select widget
depot_select = Dropdown(
    options=list_depots,
    value=list_depots[0],
    description='Depot:',
    disabled=False
)

def on_depot_select(change):
    old_layer = all_depot_layers[change['old']]
    new_layer = all_depot_layers[change['new']]
    m3.substitute(old_layer, new_layer)

depot_select.observe(on_depot_select, names='value')




# Set up interactive map
# ======================
m3 = Map(center=(region_center.y, region_center.x), zoom=16)

# Add the region border to the map
region_layer = GeoJSON(
    data=region_contour_data, 
    style={'color': 'blue', 'fillOpacity': 0.05, 'weight': 2},
    name=region_contour_data['name'])
m3.add_layer(region_layer)

# Add Voronoi polygons
voronoi_layer = GeoJSON(
    data=voronoi_data, 
    style={'color': 'blue', 'fillColor': 'lightblue', 'opacity': 0.25, 'weight': 1},
    name=voronoi_data['name'])
m3.add(voronoi_layer)

# Always keep depot points visible
depot_points = GeoJSON(
    data=depot_data,
    style={'color': 'black', 'radius':3, 'fillColor': '#3366cc', 'opacity':0.5, 'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6},
    hover_style={'fillColor': 'red' , 'fillOpacity': 0.2},
    point_style={'radius': 3, 'color': 'red', 'fillOpacity': 0.8, 'fillColor': 'blue', 'weight': 3},
    name=depot_data['name']
)
m3.add(depot_points)

# Add depot selection dropdown widget
depot_select_control = WidgetControl(widget=depot_select, position='bottomright')
m3.add(depot_select_control)

# Add (interactive + dynamic) depot layer
depot_layer = all_depot_layers[depot_select.value] # Whichever is initially set
m3.add(depot_layer)

m3.add_control(LayersControl(position='topright'))
m3.add(ScaleControl(position='bottomleft'))
m3

Map(center=[30.24893165719689, -103.6019209136208], controls=(ZoomControl(options=['position', 'zoom_in_text',…

In [13]:
depot_data

list_depots = [feature['properties']['depot_id'] for feature in depot_data['features']]
print(list_depots)

['depot_25', 'depot_93', 'depot_145']
